# Практика 4. Temporal Action Localization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Среда, данные и воспроизводимость

Официальный архив ActionFormer `thumos.tar.gz` содержит признаки и разметку.
```text
data/thumos/
├── annotations/thumos14.json
└── i3d_features/*.npy
```

Архив: [Box](https://uwmadison.box.com/s/glpuxadymf3gd01m1cj6g5c3bn39qbgr) или [Google Drive](https://drive.google.com/file/d/1zt2eoldshf99vJMDuu8jqxda55dCyhZP/view).  
MD5: `375f76ffbf7447af1035e694971ec9b2`.


In [ ]:
!git clone https://github.com/happyharrycn/actionformer_release

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import pickle
import random
import subprocess
import sys
import tarfile
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_MODE = True
TRAIN_MODEL = True
DATA_ROOT = Path("data/thumos")
ARCHIVE_PATH = Path("./thumos.tar.gz")
ACTIONFORMER_ROOT = Path("actionformer_release")
CHECKPOINT_PATH = Path("tiny_tal_head.pt")

TARGET_CLASSES = [
    "CliffDiving", "Diving", "HighJump", "LongJump",
    "PoleVault", "Shotput", "ThrowDiscus", "JavelinThrow",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: name for name, i in CLASS_TO_IDX.items()}
FEAT_STRIDE_FRAMES = 4
NUM_FRAMES = 16
STRIDES = (1, 2, 4, 8, 16, 32)
REGRESS_RANGES = ((0, 4), (4, 8), (8, 16), (16, 32), (32, 64), (64, float("inf")))
TIOU_THRESHOLDS = (0.3, 0.4, 0.5, 0.6, 0.7)

print(f"device={DEVICE}; torch={torch.__version__}; FAST_MODE={FAST_MODE}")


In [ ]:
EXPECTED_MD5 = "375f76ffbf7447af1035e694971ec9b2"

def md5sum(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.md5()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract_tar(archive: Path, destination: Path) -> None:
    destination = destination.resolve()
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Небезопасный путь в архиве: {member.name}")
        tar.extractall(destination)

if ARCHIVE_PATH.exists():
    actual = md5sum(ARCHIVE_PATH)
    print("MD5:", actual, "OK" if actual == EXPECTED_MD5 else "НЕ СОВПАЛ")
else:
    print("Архив не найден. Скачайте его по ссылке выше или укажите DATA_ROOT на распакованные данные.")

In [ ]:
!mkdir -p data/thumos
!tar -xf ./thumos.tar.gz -C data/thumos

### Формат одной записи

`VideoRecord` хранит признаки `[T, 2048]`, fps, длительность и сегменты в секундах. Важная тонкость: признак описывает окно из 16 кадров, поэтому его координата соответствует **центру окна**, а не его левому краю:

$$
t_{grid}=\frac{t_{sec}\cdot fps-0.5\cdot num\_frames}{feat\_stride}.
$$


In [ ]:
@dataclass
class VideoRecord:
    vid: str
    feats: np.ndarray
    fps: float
    duration: float
    segments: np.ndarray
    labels: np.ndarray

def seconds_to_grid(seconds: np.ndarray, fps: float) -> np.ndarray:
    # TODO: учтите центр окна I3D.
    seconds = np.asarray(seconds, dtype=np.float32)
    return (seconds * fps - 0.5 * NUM_FRAMES) / FEAT_STRIDE_FRAMES

def grid_to_seconds(grid: np.ndarray, fps: float) -> np.ndarray:
    # TODO: обратное преобразование.
    grid = np.asarray(grid, dtype=np.float32)
    return (grid * FEAT_STRIDE_FRAMES + 0.5 * NUM_FRAMES) / fps

def _normalise_features(feats: np.ndarray) -> np.ndarray:
    feats = np.asarray(feats, dtype=np.float32)
    if feats.ndim != 2:
        raise ValueError(f"Ожидались признаки [T,D], получено {feats.shape}")
    if feats.shape[0] == 2048 and feats.shape[1] != 2048:
        feats = feats.T
    if feats.shape[1] != 2048:
        raise ValueError(f"Последняя размерность должна быть 2048, получено {feats.shape}")
    return feats

def load_thumos(root: Path, split: str, classes: Sequence[str] | None = None) -> list[VideoRecord]:
    annotation_path = root / "annotations" / "thumos14.json"
    feature_dir = root / "i3d_features"
    with annotation_path.open("r", encoding="utf-8") as stream:
        payload = json.load(stream)
    database = payload.get("database", payload)
    selected = set(classes if classes is not None else TARGET_CLASSES)
    records: list[VideoRecord] = []

    for vid, info in sorted(database.items()):
        if info.get("subset") != split:
            continue
        feat_path = feature_dir / f"{vid}.npy"
        if not feat_path.exists():
            continue
        kept = []
        for ann in info.get("annotations", []):
            label = ann["label"]
            if label in selected:
                kept.append((ann["segment"], label))
        if not kept:
            continue
        feats = _normalise_features(np.load(feat_path))
        fps = float(info.get("fps", 30.0))
        duration = float(info.get("duration", grid_to_seconds(np.array([len(feats)]), fps)[0]))
        segments = np.asarray([x[0] for x in kept], dtype=np.float32)
        labels = np.asarray([CLASS_TO_IDX[x[1]] for x in kept], dtype=np.int64)
        records.append(VideoRecord(vid, feats, fps, duration, segments, labels))
    if not records:
        raise FileNotFoundError(
            f"Не найдено записей split={split!r} в {root}. Проверьте JSON, признаки и DATA_ROOT."
        )
    return records


In [ ]:
probe_seconds = np.array([0.0, 1.0, 10.0], dtype=np.float32)
probe_grid = seconds_to_grid(probe_seconds, fps=30.0)
np.testing.assert_allclose(grid_to_seconds(probe_grid, 30.0), probe_seconds, atol=1e-6)
print("round-trip seconds↔grid: OK", probe_grid)


In [ ]:
def stratified_video_subset(records, max_videos, num_classes=len(TARGET_CLASSES), seed=SEED):
    records = list(records)
    if len(records) <= max_videos:
        return records
    rng = np.random.default_rng(seed)
    label_sets = [set(map(int, record.labels.tolist())) for record in records]
    pools = {
        label: [i for i, labels in enumerate(label_sets) if label in labels]
        for label in range(num_classes)
    }
    for pool in pools.values():
        rng.shuffle(pool)

    selected = []
    selected_set = set()
    coverage = np.zeros(num_classes, dtype=int)
    while len(selected) < max_videos:
        available_classes = [
            label for label, pool in pools.items()
            if any(i not in selected_set for i in pool)
        ]
        if not available_classes:
            break
        # Сначала класс с минимальным coverage; при равенстве — более редкий в полном split.
        focus = min(available_classes, key=lambda label: (coverage[label], len(pools[label]), label))
        candidates = [i for i in pools[focus] if i not in selected_set]

        def candidate_score(index):
            # Больший score у видео, которое закрывает сразу несколько недопредставленных классов.
            return sum(1.0 / (1.0 + coverage[label]) for label in label_sets[index])

        best_score = max(candidate_score(i) for i in candidates)
        best = [i for i in candidates if candidate_score(i) == best_score]
        chosen = int(rng.choice(best))
        selected.append(chosen)
        selected_set.add(chosen)
        for label in label_sets[chosen]:
            coverage[label] += 1

    if len(selected) < max_videos:
        remaining = [i for i in range(len(records)) if i not in selected_set]
        rng.shuffle(remaining)
        selected.extend(remaining[: max_videos - len(selected)])
    return [records[i] for i in selected]

def subset_report(records):
    video_counts = np.zeros(len(TARGET_CLASSES), dtype=int)
    instance_counts = np.zeros(len(TARGET_CLASSES), dtype=int)
    for record in records:
        present = set(map(int, record.labels.tolist()))
        for label in present:
            video_counts[label] += 1
        for label in record.labels:
            instance_counts[int(label)] += 1
    return pd.DataFrame({
        "class": TARGET_CLASSES,
        "videos": video_counts,
        "instances": instance_counts,
    })

# Загружаем validation как train: это историческая особенность THUMOS14.
train_records = load_thumos(DATA_ROOT, "Validation", TARGET_CLASSES)
test_records = load_thumos(DATA_ROOT, "Test", TARGET_CLASSES)
if FAST_MODE:
    train_records = stratified_video_subset(train_records, min(32, len(train_records)), seed=SEED)
    test_records = stratified_video_subset(test_records, min(32, len(test_records)), seed=SEED + 1)
print("train videos:", len(train_records), "test videos:", len(test_records))
print("example:", train_records[0].vid, train_records[0].feats.shape)
print("train class coverage")
display(subset_report(train_records))
print("test class coverage")
display(subset_report(test_records))
if (subset_report(train_records)["videos"] == 0).any():
    missing = subset_report(train_records).query("videos == 0")["class"].tolist()
    raise RuntimeError(f"FAST train subset не покрывает классы: {missing}")


## 1. Признаки и разметка

I3D уже превратил локальные фрагменты видео в векторы. Наша модель видит только последовательность этих векторов. Сначала проверим масштаб действий относительно временного шага и посмотрим, реагирует ли норма признаков на размеченные интервалы.


In [ ]:
def dataset_stats(records: Sequence[VideoRecord]) -> pd.Series:
    durations = np.concatenate([r.segments[:, 1] - r.segments[:, 0] for r in records])
    grid_durations = np.concatenate([
        seconds_to_grid(r.segments[:, 1], r.fps) - seconds_to_grid(r.segments[:, 0], r.fps)
        for r in records
    ])
    return pd.Series({
        "videos": len(records),
        "instances": len(durations),
        "mean_sec": durations.mean(),
        "median_sec": np.median(durations),
        "mean_grid_steps": grid_durations.mean(),
        "median_grid_steps": np.median(grid_durations),
    })

display(pd.DataFrame({"train": dataset_stats(train_records), "test": dataset_stats(test_records)}))


In [ ]:
def instances_frame(records, split_name):
    rows = []
    for record in records:
        for (start, end), label in zip(record.segments, record.labels):
            rows.append({
                "split": split_name,
                "video": record.vid,
                "class": IDX_TO_CLASS[int(label)],
                "duration_sec": float(end - start),
                "duration_steps": float(
                    seconds_to_grid(np.array([end]), record.fps)[0]
                    - seconds_to_grid(np.array([start]), record.fps)[0]
                ),
                "relative_center": float((start + end) / (2 * record.duration)),
            })
    return pd.DataFrame(rows)

instances = pd.concat([
    instances_frame(train_records, "validation/train"),
    instances_frame(test_records, "test"),
], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
counts = instances.groupby(["class", "split"]).size().unstack(fill_value=0)
counts.reindex(TARGET_CLASSES).plot.bar(ax=axes[0], rot=35)
axes[0].set(title="Число экземпляров по классам", ylabel="instances")
for split_name, group in instances.groupby("split"):
    axes[1].hist(group["duration_sec"], bins=30, alpha=0.55, label=split_name)
axes[1].set(xlabel="длительность действия, с", ylabel="число", title="Распределение длительностей")
axes[1].set_yscale("log")
axes[1].legend()
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for class_name in TARGET_CLASSES:
    values = instances.loc[instances["class"] == class_name, "duration_steps"]
    if len(values):
        axes[0].hist(values, bins=np.geomspace(1, max(2, instances["duration_steps"].max()), 28),
                     histtype="step", label=class_name)
for boundary in [4, 8, 16, 32, 64]:
    axes[0].axvline(boundary, color="black", ls="--", lw=0.8, alpha=0.5)
axes[0].set_xscale("log")
axes[0].set(xlabel="длительность, шаги признаков", ylabel="instances",
            title="Длительности и границы уровней пирамиды")
axes[0].legend(fontsize=7, ncol=2)

class_order = instances.groupby("class")["duration_sec"].median().sort_values().index
box_data = [instances.loc[instances["class"] == name, "duration_sec"].to_numpy()
            for name in class_order]
axes[1].boxplot(box_data, labels=class_order, showfliers=False)
axes[1].tick_params(axis="x", rotation=35)
axes[1].set(ylabel="длительность, с", title="Масштаб действий по классам")
plt.tight_layout()


In [ ]:
def plot_record(record: VideoRecord) -> None:
    x_grid = np.arange(len(record.feats), dtype=np.float32)
    x_sec = grid_to_seconds(x_grid, record.fps)
    norms = np.linalg.norm(record.feats, axis=1)
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(14, 5), sharex=True,
                                   gridspec_kw={"height_ratios": [2, 1]})
    ax0.plot(x_sec, norms, lw=1)
    ax0.set_ylabel("||I3D feature||₂")
    for row, ((start, end), label) in enumerate(zip(record.segments, record.labels)):
        ax1.broken_barh([(start, end - start)], (row - 0.4, 0.8),
                        facecolors=f"C{int(label) % 10}")
        ax1.text(start, row, IDX_TO_CLASS[int(label)], va="center", fontsize=8)
    ax1.set_ylim(-1, max(1, len(record.segments)))
    ax1.set_xlabel("время, с")
    ax1.set_ylabel("GT")
    fig.suptitle(record.vid)
    plt.tight_layout()

plot_record(train_records[0])


In [ ]:
record = train_records[0]
sample_ids = np.linspace(0, len(record.feats) - 1, min(240, len(record.feats))).astype(int)
sampled = record.feats[sample_ids]
sampled = sampled / np.maximum(np.linalg.norm(sampled, axis=1, keepdims=True), 1e-8)
similarity = sampled @ sampled.T

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
im = axes[0].imshow(similarity, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
axes[0].set(title=f"Cosine similarity: {record.vid}", xlabel="временная позиция", ylabel="временная позиция")
fig.colorbar(im, ax=axes[0], fraction=0.046)
for split_name, group in instances.groupby("split"):
    axes[1].hist(group["relative_center"], bins=np.linspace(0, 1, 21),
                 alpha=0.55, density=True, label=split_name)
axes[1].set(xlabel="центр действия / длительность видео", ylabel="плотность",
            title="Где в видео расположены действия?")
axes[1].legend()
plt.tight_layout()


## 2. Назначение целей для anchor-free головы

На уровне с шагом `s` точка соответствует координате `(i + 0.5) · s` базовой сетки. Положительная точка лежит внутри GT, а дальняя граница попадает в диапазон масштаба уровня. Для положительной точки регрессируем `(d_start, d_end)`, нормированные на шаг уровня.

THUMOS14 допускает пересекающиеся классы (`CliffDiving` и `Diving`), поэтому классификация multi-label: sigmoid, а не softmax. Если одна точка принадлежит нескольким действиям, классификационная цель содержит несколько единиц; для регрессии выбираем самый короткий подходящий сегмент.


In [ ]:
def assign_targets(T, segments_idx, labels, num_classes=len(TARGET_CLASSES),
                   strides=STRIDES, regress_ranges=REGRESS_RANGES):
    segments_idx = np.asarray(segments_idx, dtype=np.float32).reshape(-1, 2)
    labels = np.asarray(labels, dtype=np.int64)
    result = []
    for stride, (low, high) in zip(strides, regress_ranges):
        length = math.ceil(T / stride)
        points = (np.arange(length, dtype=np.float32) + 0.5) * stride
        cls_tgt = np.zeros((length, num_classes), dtype=np.float32)
        reg_tgt = np.zeros((length, 2), dtype=np.float32)
        pos = np.zeros(length, dtype=bool)
        if len(segments_idx):
            left = points[:, None] - segments_idx[None, :, 0]
            right = segments_idx[None, :, 1] - points[:, None]
            inside = np.minimum(left, right) > 0
            max_distance = np.maximum(left, right)
            eligible = inside & (max_distance >= low) & (max_distance < high)
            lengths = segments_idx[:, 1] - segments_idx[:, 0]
            for i in range(length):
                matches = np.flatnonzero(eligible[i])
                if not len(matches):
                    continue
                pos[i] = True
                cls_tgt[i, labels[matches]] = 1.0
                chosen = matches[np.argmin(lengths[matches])]
                reg_tgt[i] = np.array([left[i, chosen], right[i, chosen]]) / stride
        result.append({"points": points, "cls": cls_tgt, "reg": reg_tgt,
                       "pos": pos, "stride": stride})
    return result


In [ ]:
toy_targets = assign_targets(64, np.array([[10, 22], [18, 40]], np.float32), np.array([0, 1]))
assert len(toy_targets) == len(STRIDES)
assert all(x["cls"].shape[1] == len(TARGET_CLASSES) for x in toy_targets)
print("positive points by level:", [int(x["pos"].sum()) for x in toy_targets])


In [ ]:
fig, axes = plt.subplots(len(STRIDES), 1, figsize=(13, 8), sharex=True)
for level, (ax, target) in enumerate(zip(axes, toy_targets)):
    ax.scatter(target["points"], np.full_like(target["points"], level),
               c=target["pos"], cmap="coolwarm", s=18)
    ax.set_ylabel(f"s={target['stride']}")
axes[-1].set_xlabel("координата базовой сетки признаков")
plt.tight_layout()


In [ ]:
positive_ratio = [target["pos"].mean() for target in toy_targets]
class_mass = np.stack([target["cls"].sum(axis=0) for target in toy_targets])
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].bar([f"s={s}" for s in STRIDES], positive_ratio)
axes[0].set(ylabel="доля positive", title="Разреженность целей по уровням")
im = axes[1].imshow(class_mass.T, aspect="auto", cmap="magma")
axes[1].set(xticks=range(len(STRIDES)), xticklabels=[f"s={s}" for s in STRIDES],
            yticks=range(len(TARGET_CLASSES)), yticklabels=TARGET_CLASSES,
            title="Число положительных меток класса")
fig.colorbar(im, ax=axes[1], label="positive labels")
plt.tight_layout()


In [ ]:
real_record = train_records[0]
real_segments_grid = seconds_to_grid(real_record.segments, real_record.fps)
real_targets = assign_targets(len(real_record.feats), real_segments_grid, real_record.labels)
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=False)
for level, target in enumerate(real_targets):
    pos_points = target["points"][target["pos"]]
    axes[0].scatter(pos_points, np.full(len(pos_points), level), s=10,
                    label=f"s={target['stride']}")
axes[0].set(ylabel="уровень пирамиды", title="Положительные точки реального видео")
axes[0].set_yticks(range(len(STRIDES)), [f"s={s}" for s in STRIDES])
level0 = real_targets[0]
axes[1].plot(level0["points"], level0["reg"][:, 0], label="d_start")
axes[1].plot(level0["points"], level0["reg"][:, 1], label="d_end")
axes[1].set(xlabel="координата сетки", ylabel="расстояние / stride",
            title="Регрессионные цели на уровне s=1")
axes[1].legend()
plt.tight_layout()


## 3. TinyTALHead

Модель сначала проецирует 2048 каналов в 256, затем создаёт шесть временных масштабов. Одна общая классификационная башня выдаёт `C` logits, регрессионная — два неотрицательных расстояния. В отличие от ActionFormer здесь нет self-attention: это намеренно простой baseline, позволяющий увидеть цену дальнего контекста.


In [ ]:
class ConvTower(nn.Module):
    def __init__(self, hidden, out_channels, depth=3):
        super().__init__()
        blocks = []
        for _ in range(depth):
            blocks += [nn.Conv1d(hidden, hidden, 3, padding=1), nn.GroupNorm(1, hidden), nn.GELU()]
        self.body = nn.Sequential(*blocks)
        self.out = nn.Conv1d(hidden, out_channels, 3, padding=1)

    def forward(self, x):
        return self.out(self.body(x))

class TinyTALHead(nn.Module):
    def __init__(self, input_dim=2048, hidden=256, num_classes=len(TARGET_CLASSES), levels=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, hidden, 3, padding=1), nn.GroupNorm(1, hidden), nn.GELU()
        )
        self.down = nn.ModuleList([
            nn.Sequential(nn.Conv1d(hidden, hidden, 3, stride=2, padding=1),
                          nn.GroupNorm(1, hidden), nn.GELU())
            for _ in range(levels - 1)
        ])
        self.cls_head = ConvTower(hidden, num_classes)
        self.reg_head = ConvTower(hidden, 2)
        prior = 0.01
        nn.init.constant_(self.cls_head.out.bias, -math.log((1 - prior) / prior))

    def forward(self, x):
        feat = self.stem(x.transpose(1, 2))
        pyramid = [feat]
        for layer in self.down:
            pyramid.append(layer(pyramid[-1]))
        cls_logits = [self.cls_head(f).transpose(1, 2) for f in pyramid]
        reg_out = [F.softplus(self.reg_head(f)).transpose(1, 2) for f in pyramid]
        return cls_logits, reg_out


In [ ]:
probe_model = TinyTALHead(hidden=64).to(DEVICE)
with torch.no_grad():
    probe_cls, probe_reg = probe_model(torch.randn(2, 129, 2048, device=DEVICE))
print("cls shapes:", [tuple(x.shape) for x in probe_cls])
print("reg shapes:", [tuple(x.shape) for x in probe_reg])
assert all(c.shape[:2] == r.shape[:2] for c, r in zip(probe_cls, probe_reg))
del probe_model, probe_cls, probe_reg


In [ ]:
level_lengths = [math.ceil(129 / s) for s in STRIDES]
total_candidates = [length * len(TARGET_CLASSES) for length in level_lengths]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar([f"s={s}" for s in STRIDES], level_lengths)
axes[0].set(title="Длина временной пирамиды при T=129", ylabel="точки")
axes[1].bar([f"s={s}" for s in STRIDES], total_candidates)
axes[1].set(title="Число point×class кандидатов", ylabel="кандидаты")
plt.tight_layout()


### Focal loss и 1D-DIoU

Focal loss уменьшает вклад многочисленных простых фоновых точек. Регрессия сравнивает декодированные интервалы: `1 − tIoU` штрафует несовпадение, а расстояние между центрами делит на квадрат длины минимального охватывающего интервала.


In [ ]:
def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    prob = logits.sigmoid()
    p_t = prob * targets + (1 - prob) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * (1 - p_t).pow(gamma) * ce

def diou_1d_loss(points, pred_dist, target_dist, stride):
    pred_start = points - pred_dist[:, 0] * stride
    pred_end = points + pred_dist[:, 1] * stride
    tgt_start = points - target_dist[:, 0] * stride
    tgt_end = points + target_dist[:, 1] * stride
    inter = (torch.minimum(pred_end, tgt_end) - torch.maximum(pred_start, tgt_start)).clamp_min(0)
    union = (pred_end - pred_start) + (tgt_end - tgt_start) - inter
    iou = inter / union.clamp_min(1e-6)
    pred_center = 0.5 * (pred_start + pred_end)
    tgt_center = 0.5 * (tgt_start + tgt_end)
    cover = (torch.maximum(pred_end, tgt_end) - torch.minimum(pred_start, tgt_start)).clamp_min(1e-6)
    return 1 - iou + (pred_center - tgt_center).pow(2) / cover.pow(2)


In [ ]:
p = torch.tensor([10.0])
target_d = torch.tensor([[2.0, 3.0]])
assert torch.allclose(diou_1d_loss(p, target_d, target_d, 1), torch.zeros(1), atol=1e-6)
assert diou_1d_loss(p, torch.tensor([[1.0, 4.0]]), target_d, 1).item() > 0
print("loss unit tests: OK")


## 4. Кропы, batching и обучение

Длинные последовательности случайно обрезаются до `max_len`. Сегменты переводятся в сетку и сдвигаются относительно окна. В батче последовательности дополняются нулями, а mask исключает padding из loss.


In [ ]:
def crop_record(record: VideoRecord, max_len: int, rng: np.random.Generator):
    T = len(record.feats)
    segments = seconds_to_grid(record.segments, record.fps)
    if T <= max_len:
        return record.feats, segments, record.labels
    if len(segments) and rng.random() < 0.7:
        chosen = segments[rng.integers(len(segments))]
        center = 0.5 * (chosen[0] + chosen[1])
        start = int(np.clip(center - rng.uniform(0.25, 0.75) * max_len, 0, T - max_len))
    else:
        start = int(rng.integers(0, T - max_len + 1))
    end = start + max_len
    overlap = np.minimum(segments[:, 1], end) - np.maximum(segments[:, 0], start)
    keep = overlap > 0
    cropped = segments[keep].copy()
    cropped[:, 0] = np.clip(cropped[:, 0] - start, 0, max_len)
    cropped[:, 1] = np.clip(cropped[:, 1] - start, 0, max_len)
    return record.feats[start:end], cropped, record.labels[keep]

def make_batch(records, indices, max_len, rng):
    samples = [crop_record(records[i], max_len, rng) for i in indices]
    lengths = [len(x[0]) for x in samples]
    padded_len = max(lengths)
    feats = np.zeros((len(samples), padded_len, samples[0][0].shape[1]), np.float32)
    for i, (x, _, _) in enumerate(samples):
        feats[i, : len(x)] = x
    return torch.from_numpy(feats), lengths, [(x[1], x[2]) for x in samples]


In [ ]:
def tal_loss(cls_logits, reg_out, lengths, annotations):
    cls_sum = torch.zeros((), device=cls_logits[0].device)
    reg_sum = torch.zeros((), device=cls_logits[0].device)
    positives = 0
    for b, (length, (segments, labels)) in enumerate(zip(lengths, annotations)):
        targets = assign_targets(length, segments, labels)
        for level, (cls_level, reg_level, target) in enumerate(zip(cls_logits, reg_out, targets)):
            valid_len = math.ceil(length / STRIDES[level])
            logits = cls_level[b, :valid_len]
            pred_reg = reg_level[b, :valid_len]
            cls_tgt = torch.as_tensor(target["cls"][:valid_len], device=logits.device)
            reg_tgt = torch.as_tensor(target["reg"][:valid_len], device=logits.device)
            pos = torch.as_tensor(target["pos"][:valid_len], device=logits.device)
            cls_sum = cls_sum + sigmoid_focal_loss(logits, cls_tgt).sum()
            n_pos = int(pos.sum().item())
            positives += n_pos
            if n_pos:
                points = torch.as_tensor(target["points"][:valid_len], device=logits.device)
                reg_sum = reg_sum + diou_1d_loss(
                    points[pos], pred_reg[pos], reg_tgt[pos], STRIDES[level]
                ).sum()
    normalizer = max(positives, 1)
    cls_loss = cls_sum / normalizer
    reg_loss = reg_sum / normalizer
    return cls_loss + reg_loss, {"focal": cls_loss.item(), "diou": reg_loss.item(), "positive": positives}


In [ ]:
def train_tal(model, records, epochs, lr=1e-3, batch_size=2, max_len=2304):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    warmup_epochs = min(5, max(1, epochs // 5))
    def lr_factor(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    rng = np.random.default_rng(SEED)
    history = []
    for epoch in range(epochs):
        model.train()
        order = rng.permutation(len(records))
        rows = []
        for start in tqdm(range(0, len(order), batch_size), leave=False, desc=f"epoch {epoch+1}"):
            ids = order[start:start + batch_size]
            feats, lengths, annotations = make_batch(records, ids, max_len, rng)
            feats = feats.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            cls_logits, reg_out = model(feats)
            loss, parts = tal_loss(cls_logits, reg_out, lengths, annotations)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            rows.append({"loss": loss.item(), **parts})
        scheduler.step()
        summary = pd.DataFrame(rows).mean(numeric_only=True).to_dict()
        summary.update(epoch=epoch + 1, lr=optimizer.param_groups[0]["lr"])
        history.append(summary)
        print({k: round(v, 4) for k, v in summary.items()})
    return pd.DataFrame(history)

model = TinyTALHead().to(DEVICE)
EPOCHS = 30 if FAST_MODE else 30
if TRAIN_MODEL:
    history = train_tal(model, train_records, epochs=EPOCHS,
                        batch_size=2, max_len=768 if FAST_MODE else 2304)
    torch.save({"state_dict": model.state_dict(), "classes": TARGET_CLASSES}, CHECKPOINT_PATH)
elif CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    history = pd.DataFrame()
else:
    raise FileNotFoundError("Нет checkpoint: включите TRAIN_MODEL или укажите CHECKPOINT_PATH")


In [ ]:
if len(history):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    history.plot(x="epoch", y=["focal", "diou"], marker="o", grid=True, ax=axes[0])
    axes[0].set(ylabel="loss / positive point", title="Компоненты функции потерь")
    history.plot(x="epoch", y="lr", marker="o", grid=True, ax=axes[1], legend=False)
    axes[1].set(ylabel="learning rate", title="Warm-up и cosine schedule")
    history.plot(x="epoch", y="positive", marker="o", grid=True, ax=axes[2], legend=False)
    axes[2].set(ylabel="positive points / epoch mean", title="Сколько сигнала видит модель")
    plt.tight_layout()
    plt.show()


## 5. Декодирование и подавление дублей

Каждая пара `(point, class)` становится кандидатом. Расстояния возвращают границы базовой сетки, затем координаты переводятся в секунды. После порога и `pre_nms_topk` остаётся много почти одинаковых сегментов, поэтому применяем NMS отдельно по классам.


In [ ]:
def temporal_iou_one_to_many(segment, others):
    segment = np.asarray(segment, np.float32)
    others = np.asarray(others, np.float32).reshape(-1, 2)
    inter = np.maximum(0, np.minimum(segment[1], others[:, 1]) - np.maximum(segment[0], others[:, 0]))
    union = (segment[1] - segment[0]) + (others[:, 1] - others[:, 0]) - inter
    return inter / np.maximum(union, 1e-8)

def soft_nms(detections, sigma=0.5, min_score=0.001, max_det=200):
    kept = []
    for label in sorted({d[2] for d in detections}):
        pool = [list(d) for d in detections if d[2] == label]
        class_kept = []
        while pool and len(class_kept) < max_det:
            best_idx = int(np.argmax([d[3] for d in pool]))
            best = pool.pop(best_idx)
            class_kept.append(tuple(best))
            if not pool:
                break
            overlaps = temporal_iou_one_to_many(best[:2], [d[:2] for d in pool])
            new_pool = []
            for det, overlap in zip(pool, overlaps):
                det[3] *= math.exp(-(float(overlap) ** 2) / sigma)
                if det[3] >= min_score:
                    new_pool.append(det)
            pool = new_pool
        kept.extend(class_kept)
    return sorted(kept, key=lambda x: x[3], reverse=True)[:max_det]


In [ ]:
def hard_nms(detections, threshold=0.5, max_det=200):
    kept = []
    for label in sorted({d[2] for d in detections}):
        pool = sorted([d for d in detections if d[2] == label], key=lambda x: x[3], reverse=True)
        while pool and len(kept) < max_det:
            best = pool.pop(0)
            kept.append(best)
            if pool:
                overlaps = temporal_iou_one_to_many(best[:2], [d[:2] for d in pool])
                pool = [d for d, ov in zip(pool, overlaps) if ov <= threshold]
    return sorted(kept, key=lambda x: x[3], reverse=True)[:max_det]

@torch.inference_mode()
def decode_record(model, record, score_thr=0.001, pre_nms_topk=2000, nms="soft"):
    model.eval()
    x = torch.from_numpy(record.feats[None]).to(DEVICE)
    cls_levels, reg_levels = model(x)
    candidates = []
    for level, (logits, distances, stride) in enumerate(zip(cls_levels, reg_levels, STRIDES)):
        scores = logits[0].sigmoid().cpu().numpy()
        reg = distances[0].cpu().numpy()
        points = (np.arange(len(scores), dtype=np.float32) + 0.5) * stride
        flat = scores.ravel()
        ids = np.flatnonzero(flat >= score_thr)
        if len(ids) > pre_nms_topk:
            ids = ids[np.argpartition(flat[ids], -pre_nms_topk)[-pre_nms_topk:]]
        for flat_idx in ids:
            point_idx, label = np.unravel_index(flat_idx, scores.shape)
            start_grid = points[point_idx] - reg[point_idx, 0] * stride
            end_grid = points[point_idx] + reg[point_idx, 1] * stride
            start, end = grid_to_seconds(np.array([start_grid, end_grid]), record.fps)
            start = float(np.clip(start, 0, record.duration))
            end = float(np.clip(end, 0, record.duration))
            if end > start:
                candidates.append((start, end, int(label), float(scores[point_idx, label])))
    if nms == "soft":
        return candidates, soft_nms(candidates)
    if nms == "hard":
        return candidates, hard_nms(candidates)
    return candidates, candidates

raw_example, soft_example = decode_record(model, test_records[0], nms="soft")
_, hard_example = decode_record(model, test_records[0], nms="hard")
print("raw / hard / soft:", len(raw_example), len(hard_example), len(soft_example))


In [ ]:
def plot_detections(record, detections, title, topk=30):
    fig, ax = plt.subplots(figsize=(14, 4))
    for row, (start, end, label, score) in enumerate(detections[:topk]):
        ax.broken_barh([(start, end-start)], (row-0.35, 0.7), facecolors=f"C{label%10}", alpha=0.8)
        ax.text(start, row, f"{IDX_TO_CLASS[label]} {score:.2f}", va="center", fontsize=7)
    for start, end in record.segments:
        ax.axvspan(start, end, color="black", alpha=0.06)
    ax.set(title=title, xlabel="время, с", ylabel="prediction rank")
    plt.tight_layout()

plot_detections(test_records[0], raw_example, "До NMS")
plot_detections(test_records[0], soft_example, "После Soft-NMS")


In [ ]:
def overlay_nms(record, variants, topk=20):
    fig, axes = plt.subplots(len(variants), 1, figsize=(15, 2.7 * len(variants)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, (name, detections) in zip(axes, variants.items()):
        for row, (start, end, label, score) in enumerate(detections[:topk]):
            ax.plot([start, end], [row, row], lw=4, alpha=max(0.2, min(1, score * 4)),
                    color=f"C{label % 10}")
        for start, end in record.segments:
            ax.axvspan(start, end, color="black", alpha=0.06)
        ax.set(ylabel=name, title=f"{name}: top-{min(topk, len(detections))}")
    axes[-1].set_xlabel("время, с")
    plt.tight_layout()

overlay_nms(test_records[0], {"raw": raw_example, "hard": hard_example, "soft": soft_example})

overlaps = np.linspace(0, 1, 101)
plt.figure(figsize=(7, 4))
plt.plot(overlaps, np.exp(-(overlaps ** 2) / 0.5), label="Soft-NMS, σ=0.5")
plt.plot(overlaps, (overlaps <= 0.5).astype(float), label="hard NMS, τ=0.5")
plt.xlabel("tIoU с лучшим сегментом")
plt.ylabel("множитель score")
plt.title("Как NMS реагирует на перекрытие")
plt.grid(True)
plt.legend()
plt.show()


## 6. mAP@tIoU

Для каждого класса предсказания всех видео сортируются по score. Предсказание может занять только один ещё свободный GT того же видео. Дубликат — FP. После накопленных TP/FP строится precision–recall, precision интерполируется проходом справа налево, AP равна площади ступенчатой кривой.


In [ ]:
def tiou(seg_a, seg_b):
    a = np.asarray(seg_a, np.float32).reshape(-1, 2)
    b = np.asarray(seg_b, np.float32).reshape(-1, 2)
    inter = np.maximum(0, np.minimum(a[:, None, 1], b[None, :, 1]) -
                          np.maximum(a[:, None, 0], b[None, :, 0]))
    len_a = np.maximum(0, a[:, 1] - a[:, 0])[:, None]
    len_b = np.maximum(0, b[:, 1] - b[:, 0])[None, :]
    return inter / np.maximum(len_a + len_b - inter, 1e-8)

def interpolated_ap(tp, fp, n_gt):
    if n_gt == 0:
        return np.nan, np.array([]), np.array([])
    tp_cum, fp_cum = np.cumsum(tp), np.cumsum(fp)
    recall = tp_cum / n_gt
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
    mrec = np.r_[0.0, recall, 1.0]
    mpre = np.r_[0.0, precision, 0.0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    changes = np.flatnonzero(mrec[1:] != mrec[:-1]) + 1
    ap = np.sum((mrec[changes] - mrec[changes - 1]) * mpre[changes])
    return float(ap), precision, recall

def compute_map(preds, gts, thresholds=TIOU_THRESHOLDS, classes=range(len(TARGET_CLASSES))):
    rows = []
    for threshold in thresholds:
        for label in classes:
            gt_by_video = {
                vid: np.asarray([[d[0], d[1]] for d in items if d[2] == label], np.float32).reshape(-1, 2)
                for vid, items in gts.items()
            }
            n_gt = sum(len(x) for x in gt_by_video.values())
            occupied = {vid: np.zeros(len(x), dtype=bool) for vid, x in gt_by_video.items()}
            ranked = sorted(
                [(vid, d) for vid, items in preds.items() for d in items if d[2] == label],
                key=lambda x: x[1][3], reverse=True,
            )
            tp = np.zeros(len(ranked), np.float32)
            fp = np.zeros(len(ranked), np.float32)
            for i, (vid, det) in enumerate(ranked):
                candidates = gt_by_video.get(vid, np.empty((0, 2), np.float32))
                free = np.flatnonzero(~occupied.get(vid, np.zeros(0, bool)))
                if not len(free):
                    fp[i] = 1
                    continue
                overlaps = tiou(np.asarray(det[:2])[None], candidates[free])[0]
                best_local = int(np.argmax(overlaps))
                if overlaps[best_local] >= threshold:
                    occupied[vid][free[best_local]] = True
                    tp[i] = 1
                else:
                    fp[i] = 1
            ap, precision, recall = interpolated_ap(tp, fp, n_gt)
            rows.append({"threshold": threshold, "label": label, "ap": ap,
                         "n_gt": n_gt, "precision": precision, "recall": recall})
    per_class = pd.DataFrame(rows)
    summary = per_class.groupby("threshold", as_index=False)["ap"].mean()
    summary["metric"] = summary["threshold"].map(lambda x: f"mAP@{x:.1f}")
    average = float(summary["ap"].mean())
    return summary[["metric", "threshold", "ap"]], per_class, average


In [ ]:
toy_gts = {"v": [(0, 10, 0), (20, 30, 0), (40, 50, 0), (60, 70, 0)]}
toy_preds = {"v": [
    (1.0, 11.0, 0, 0.95),
    (22.35, 32.35, 0, 0.90),
    (64.49, 74.49, 0, 0.80),
    (1.235, 11.235, 0, 0.70),
    (42.82, 52.82, 0, 0.60),
]}
toy_summary, toy_per_class, toy_average = compute_map(toy_preds, toy_gts, thresholds=(0.5,), classes=(0,))
display(toy_summary)
assert abs(toy_average - 0.65) < 0.01, toy_average


In [ ]:
def records_to_gts(records):
    return {
        r.vid: [(float(s), float(e), int(label))
                for (s, e), label in zip(r.segments, r.labels)]
        for r in records
    }

@torch.inference_mode()
def predict_dataset(model, records, nms):
    predictions = {}
    for record in tqdm(records, desc=f"inference/{nms}"):
        raw, processed = decode_record(model, record, nms=nms)
        predictions[record.vid] = processed
    return predictions

gts_test = records_to_gts(test_records)
preds_soft = predict_dataset(model, test_records, "soft")
preds_hard = predict_dataset(model, test_records, "hard")
preds_none = predict_dataset(model, test_records, "none")

map_soft, per_class_soft, avg_soft = compute_map(preds_soft, gts_test)
map_hard, _, avg_hard = compute_map(preds_hard, gts_test)
map_none, _, avg_none = compute_map(preds_none, gts_test)
comparison = pd.DataFrame({
    "tIoU": map_soft["threshold"],
    "none": map_none["ap"], "hard NMS": map_hard["ap"], "Soft-NMS": map_soft["ap"],
})
display(comparison.style.format("{:.3f}"))
print({"avg_none": avg_none, "avg_hard": avg_hard, "avg_soft": avg_soft})


In [ ]:
ap_matrix = per_class_soft.pivot(index="label", columns="threshold", values="ap")
ap_matrix = ap_matrix.reindex(range(len(TARGET_CLASSES)))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
im = axes[0].imshow(ap_matrix.to_numpy(), vmin=0, vmax=1, cmap="viridis", aspect="auto")
axes[0].set(
    xticks=range(len(ap_matrix.columns)),
    xticklabels=[f"{x:.1f}" for x in ap_matrix.columns],
    yticks=range(len(TARGET_CLASSES)), yticklabels=TARGET_CLASSES,
    xlabel="tIoU", title="AP по классам и строгости границ",
)
fig.colorbar(im, ax=axes[0], label="AP")

duration_by_class = instances.groupby("class")["duration_sec"].median()
ap_drop = ap_matrix.iloc[:, 0] - ap_matrix.iloc[:, -1]
x = np.asarray([duration_by_class.get(IDX_TO_CLASS[i], np.nan) for i in ap_matrix.index])
axes[1].scatter(x, ap_drop.to_numpy(), s=60)
for i, (duration, drop) in enumerate(zip(x, ap_drop.to_numpy())):
    if np.isfinite(duration) and np.isfinite(drop):
        axes[1].annotate(IDX_TO_CLASS[i], (duration, drop), fontsize=8)
axes[1].set(xlabel="медианная длительность класса, с", ylabel="AP@0.3 − AP@0.7",
            title="Каким классам труднее поставить точные границы?")
axes[1].grid(True)
plt.tight_layout()


## 7. ActionFormer

Сначала подаём наши предсказания в официальный `ANETdetection`. Затем при наличии клона и checkpoint запускаем полный ActionFormer.


In [ ]:
# !pip install nms_1d_cpu

In [ ]:
def make_subset_eval_json(source_path, destination_path, records, class_names):
    with Path(source_path).open("r", encoding="utf-8") as stream:
        payload = json.load(stream)
    allowed_videos = {r.vid for r in records}
    database = payload["database"]
    subset_db = {}
    name_to_official_id = {}
    for vid, info in database.items():
        if vid not in allowed_videos:
            continue
        item = dict(info)
        item["annotations"] = [a for a in info.get("annotations", []) if a["label"] in class_names]
        for ann in item["annotations"]:
            name_to_official_id[ann["label"]] = int(ann["label_id"])
        if item["annotations"]:
            subset_db[vid] = item
    subset_payload = dict(payload)
    subset_payload["database"] = subset_db
    Path(destination_path).write_text(json.dumps(subset_payload, ensure_ascii=False), encoding="utf-8")
    return name_to_official_id

def to_actionformer_predictions(preds, name_to_official_id):
    rows = [(vid, s, e, label, score)
            for vid, items in preds.items() for s, e, label, score in items]
    if not rows:
        return {"video-id": [], "t-start": np.array([]), "t-end": np.array([]),
                "label": np.array([], dtype=int), "score": np.array([])}
    return {
        "video-id": [x[0] for x in rows],
        "t-start": np.asarray([x[1] for x in rows]),
        "t-end": np.asarray([x[2] for x in rows]),
        "label": np.asarray([name_to_official_id[IDX_TO_CLASS[x[3]]] for x in rows]),
        "score": np.asarray([x[4] for x in rows]),
    }

if ACTIONFORMER_ROOT.exists():
    # Не импортируем `libs.utils.metrics` обычным способом: пакет `libs.utils`
    # заодно импортирует compiled NMS, который для вычисления метрики не нужен.
    import importlib.util
    metrics_path = ACTIONFORMER_ROOT / "libs" / "utils" / "metrics.py"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Не найден официальный evaluator: {metrics_path.resolve()}")
    spec = importlib.util.spec_from_file_location("actionformer_metrics_standalone", metrics_path)
    metrics_module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(metrics_module)
    ANETdetection = metrics_module.ANETdetection
    # Старые версии official code используют удалённый в NumPy 2 алиас np.float.
    if not hasattr(np, "float"):
        np.float = float
    subset_eval_path = Path("thumos14_subset8_eval.json")
    official_ids = make_subset_eval_json(
        DATA_ROOT / "annotations" / "thumos14.json", subset_eval_path,
        test_records, TARGET_CLASSES,
    )
    evaluator = ANETdetection(
        ant_file=str(subset_eval_path.resolve()),
        split="test", tiou_thresholds=np.asarray(TIOU_THRESHOLDS), num_workers=1,
    )
    official_predictions = to_actionformer_predictions(preds_soft, official_ids)
    predicted_ids = set(np.asarray(official_predictions["label"]).tolist())
    missing_names = sorted(name for name, label_id in official_ids.items()
                           if label_id not in predicted_ids)
    if missing_names:
        print(
            "Нет предсказаний после threshold/NMS для классов:", missing_names,
            "— official evaluator напечатает warnings и присвоит им AP=0."
        )
    official = evaluator.evaluate(official_predictions, verbose=True)
    print("official evaluator output:", official)
    custom_values = map_soft["ap"].to_numpy()
    max_diff = float(np.max(np.abs(official[0] - custom_values)))
    max_diff_pp = 100 * max_diff
    # Критерий из задания: расхождение меньше 0.01 процентного пункта.
    tolerance_pp = 0.01
    if max_diff_pp >= tolerance_pp:
        raise AssertionError(
            f"Метрики расходятся на {max_diff_pp:.6f} п.п.; "
            f"допустимо < {tolerance_pp:.2f} п.п. Проверьте matching, dtype и одинаковый JSON."
        )
    print(
        f"Сверка пройдена: максимальное расхождение {max_diff_pp:.6f} п.п. "
        f"(< {tolerance_pp:.2f} п.п.)."
    )
else:
    print("Пропуск: клонируйте https://github.com/happyharrycn/actionformer_release")


In [ ]:
ACTIONFORMER_CONFIG = ACTIONFORMER_ROOT / "configs" / "thumos_i3d.yaml"
ACTIONFORMER_CKPT_DIR = ACTIONFORMER_ROOT / "pretrained" / "thumos_i3d_reproduce"
RUN_ACTIONFORMER = False  # включите после подготовки клона, данных и checkpoint

if RUN_ACTIONFORMER:
    if not ACTIONFORMER_CONFIG.exists() or not ACTIONFORMER_CKPT_DIR.exists():
        raise FileNotFoundError("Не найдены config/checkpoint ActionFormer")
    command = [sys.executable, "eval.py", str(ACTIONFORMER_CONFIG.resolve()),
               str(ACTIONFORMER_CKPT_DIR.resolve()), "--saveonly"]
    subprocess.run(command, cwd=ACTIONFORMER_ROOT, check=True)
else:
    print("Команда:", sys.executable, "eval.py", ACTIONFORMER_CONFIG, ACTIONFORMER_CKPT_DIR, "--saveonly")


In [ ]:
ACTIONFORMER_RESULTS = ACTIONFORMER_CKPT_DIR / "eval_results.pkl"

def load_actionformer_predictions(path, annotation_path, allowed_videos=None):
    with Path(path).open("rb") as stream:
        raw = pickle.load(stream)
    with Path(annotation_path).open("r", encoding="utf-8") as stream:
        database = json.load(stream)["database"]
    official_id_to_name = {}
    for info in database.values():
        for ann in info.get("annotations", []):
            official_id_to_name[int(ann["label_id"])] = ann["label"]
    allowed_videos = set(allowed_videos) if allowed_videos is not None else None
    converted = {}
    for vid, start, end, official_label, score in zip(
        raw["video-id"], raw["t-start"], raw["t-end"], raw["label"], raw["score"]
    ):
        name = official_id_to_name[int(official_label)]
        if name not in CLASS_TO_IDX or (allowed_videos is not None and vid not in allowed_videos):
            continue
        converted.setdefault(vid, []).append(
            (float(start), float(end), CLASS_TO_IDX[name], float(score))
        )
    if allowed_videos is not None:
        for vid in allowed_videos:
            converted.setdefault(vid, [])
    return converted

if ACTIONFORMER_RESULTS.exists():
    preds_actionformer = load_actionformer_predictions(
        ACTIONFORMER_RESULTS,
        DATA_ROOT / "annotations" / "thumos14.json",
        allowed_videos=gts_test,
    )
    map_actionformer, per_class_actionformer, avg_actionformer = compute_map(
        preds_actionformer, gts_test
    )
    display(map_actionformer.style.format({"ap": "{:.3f}"}))
    print("ActionFormer average mAP on the same subset:", round(avg_actionformer, 4))
else:
    print("После запуска с --saveonly здесь появится", ACTIONFORMER_RESULTS)




1. Постройте `mAP(tIoU)` для модели без NMS, с hard NMS и Soft-NMS.  
2. Для одного класса постройте precision–recall при `tIoU=0.5`.  



In [ ]:
ax = comparison.plot(x="tIoU", y=["none", "hard NMS", "Soft-NMS"],
                     marker="o", figsize=(9, 5), grid=True)
ax.set(ylabel="mAP", title="Точность границ и влияние NMS")
plt.show()

chosen_label = 0
row = per_class_soft.query("threshold == 0.5 and label == @chosen_label").iloc[0]
plt.figure(figsize=(6, 5))
plt.step(row["recall"], row["precision"], where="post")
plt.xlabel("recall")
plt.ylabel("precision")
plt.title(f"PR: {IDX_TO_CLASS[chosen_label]}, tIoU=0.5")
plt.grid(True)
plt.show()


## Дополнительные эксперименты

- Повторить обучение на всех 20 классах.
- Заменить I3D на VideoMAE/VideoMAEv2 при неизменной голове.
- Добавить center sampling и EMA-нормировку числа положительных точек, как в ActionFormer.
- Сравнить Conv1d-блок с локальным self-attention.
- Провести ablation по числу уровней, `sigma` Soft-NMS и диапазонам регрессии.

## Источники

- Zhang, Wu, Li. *ActionFormer: Localizing Moments of Actions with Transformers*, ECCV 2022.
- Официальный репозиторий: https://github.com/happyharrycn/actionformer_release
- THUMOS Challenge 2014: https://www.crcv.ucf.edu/THUMOS14/
- OpenTAD: https://github.com/sming256/OpenTAD
